In [0]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Change only these if your Unity Catalog names are different
CATALOG = f"formula1_dev"
BRONZE = "bronze"
SILVER = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}")

def bronze_table(name):
    return f"{CATALOG}.{BRONZE}.{name}"

def silver_table(name):
    return f"{CATALOG}.{SILVER}.{name}"

In [0]:
qualifying=spark.table(bronze_table("qualifying"))
qualifying.printSchema()
print("Bronze rows:",qualifying.count())

## 1. NULL + duplicate checks

In [0]:
display(qualifying.filter(
    F.col("qualify_id").isNull() |
    F.col("race_id").isNull() |
    F.col("driver_id").isNull()
))
display(qualifying.groupBy("qualify_id").count().filter(F.col("count")>1))

## 2. String operations

In [0]:
qual_work=(
    qualifying
    .withColumn("q1_clean",F.trim("q1"))
    .withColumn("q2_clean",F.trim("q2"))
    .withColumn("q3_clean",F.trim("q3"))
    .withColumn("q1_length",F.length("q1_clean"))
    .withColumn("q1_parts",F.split("q1_clean",":"))
    .withColumn("q1_prefix",F.substring("q1_clean",1,3))
)
display(qual_work.select(
    "qualify_id","q1","q1_clean","q1_length","q1_parts","q1_prefix"
).limit(20))

## 3. contains / startsWith / endsWith

In [0]:
display(qual_work.filter(F.col("q1_clean").contains(":"))
                  .select("qualify_id","q1_clean").limit(20))
display(qual_work.filter(F.col("q1_clean").startswith("1"))
                  .select("qualify_id","q1_clean").limit(20))
display(qual_work.filter(F.col("q1_clean").endswith("0"))
                  .select("qualify_id","q1_clean").limit(20))

## 4. Incremental batch

In [0]:
target_name=silver_table("qualifying")

if not spark.catalog.tableExists(target_name):
    qualifying_batch=qualifying
else:
    last_ts=spark.table(target_name).agg(
        F.max("ingestion_timestamp").alias("max_ts")
    ).first()["max_ts"]

    print("Last Silver ingestion_timestamp:",last_ts)

    qualifying_batch=(
        qualifying if last_ts is None
        else qualifying.filter(F.col("ingestion_timestamp")>F.lit(last_ts))
    )

print("Rows selected:",qualifying_batch.count())

## 5. Clean selected batch

In [0]:
qualifying_clean=(
    qualifying_batch
    .filter(F.col("qualify_id").isNotNull())
    .filter(F.col("race_id").isNotNull())
    .filter(F.col("driver_id").isNotNull())
    .dropDuplicates(["qualify_id"])
    .withColumn("q1_clean",F.trim("q1"))
    .withColumn("q2_clean",F.trim("q2"))
    .withColumn("q3_clean",F.trim("q3"))
    .withColumn("position_int",F.col("position").cast("int"))
    .withColumn(
        "qualifying_category",
        F.when(F.col("position_int")==1,"Pole")
         .when(F.col("position_int").between(2,3),"Front Row")
         .when(F.col("position_int").between(4,10),"Top 10")
         .otherwise("Outside Top 10")
    )
    .withColumn("silver_processed_timestamp",F.current_timestamp())
)
display(qualifying_clean.limit(20))

## 6. MERGE

In [0]:
if not spark.catalog.tableExists(target_name):
    (qualifying_clean.write.format("delta").mode("overwrite")
     .option("overwriteSchema","true").saveAsTable(target_name))
else:
    target=DeltaTable.forName(spark,target_name)
    (target.alias("t")
     .merge(qualifying_clean.alias("s"),"t.qualify_id=s.qualify_id")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())

print("QUALIFYING MERGE completed.")

In [0]:
silver=spark.table(target_name)
print("Silver rows:",silver.count())
display(silver.orderBy(F.desc("ingestion_timestamp")).limit(20))